# Legacy shield-aware surface MLE

This self-contained example preserves the useful experimental ideas from `estimation_sub.ipynb`, `estimation_3.ipynb`, and `estimation_5.ipynb` in the predecessor repository [`moeuu/Radiation_distribution_machine_learning`](https://github.com/moeuu/Radiation_distribution_machine_learning) at commit `d07b4ead`. No predecessor CSV data, copied optimizer, or duplicated shield physics is retained.

The example uses the shared simulation runtime's `RuntimeObservationModel` and `ContinuousKernel` to build count responses, then calls the current `fit_surface_map_poisson` solver. It compares a shield-aware response with a zero-shield ablation for one-, three-, and five-source scenes. Source strength is detector count rate at 1 m (`cps@1m`), not total gamma activity.

> The sampled observations below are deterministic synthetic examples, not production observations or a reproduction of the original thesis measurements. Production estimation must consume a simulator-generated or imported MeasurementLog v2.

Launch with `uv run --with jupyter jupyter lab` from the repository root.

In [ ]:
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np

from measurement.model import EnvironmentConfig
from measurement.observation_model import (
    build_runtime_observation_model,
    continuous_kernel_from_observation_model,
)
from three_d_estimation import (
    SurfaceMapConfig,
    build_count_response,
    build_surface_patches,
    fit_surface_map_poisson,
)
from three_d_estimation.postprocess import cluster_surface_hotspots

## Experiment definition

The source coordinates and strengths retain the three useful historical scenarios. The floor, observation schedule, count data, and responses are generated in memory. Four downward-facing Fe/Pb orientation pairs provide the directional shield views.

In [ ]:
ROOM_SIZE_M = 10.0
ROOM_HEIGHT_M = 3.0
GRID_SPACING_M = 1.0
DETECTOR_HEIGHT_M = 0.75
DWELL_TIME_S = 30.0
BACKGROUND_CPS = 1.0
ISOTOPES = ("Cs-137",)
DOWNWARD_VIEW_INDICES = np.asarray([1, 3, 5, 7], dtype=np.int64)
RANDOM_SEED = 20240208

SCENARIOS = {
    "one_source": (
        (2.0, 6.5, 90.0),
    ),
    "three_sources": (
        (2.0, 2.0, 50.0),
        (5.5, 5.5, 80.0),
        (8.0, 6.0, 90.0),
    ),
    "five_sources": (
        (1.5, 1.5, 50.0),
        (8.5, 1.5, 80.0),
        (1.5, 8.5, 30.0),
        (8.5, 8.5, 70.0),
        (4.5, 4.5, 90.0),
    ),
}

In [ ]:
def build_floor_grid():
    """Build one-metre floor patches with the current surface builder."""
    complete = build_surface_patches(
        EnvironmentConfig(ROOM_SIZE_M, ROOM_SIZE_M, ROOM_HEIGHT_M),
        obstacle_grid=None,
        spacing=(GRID_SPACING_M, GRID_SPACING_M, ROOM_HEIGHT_M),
        quadrature_points_per_patch=1,
    )
    selected = np.flatnonzero(np.asarray(complete.kinds) == "floor")
    remap = {int(old): new for new, old in enumerate(selected)}
    edges = []
    edge_weights = []
    for edge, weight in zip(
        complete.adjacency_index_edges,
        complete.adjacency_weights,
        strict=True,
    ):
        first, second = int(edge[0]), int(edge[1])
        if first in remap and second in remap:
            edges.append((remap[first], remap[second]))
            edge_weights.append(float(weight))

    return SimpleNamespace(
        areas_m2=complete.areas_m2[selected],
        centroids_xyz=complete.centroids_xyz[selected],
        kinds=tuple(np.asarray(complete.kinds)[selected].tolist()),
        patch_ids=np.arange(selected.size, dtype=np.int64),
        quadrature_points_xyz=complete.quadrature_points_xyz[selected],
        quadrature_weights=complete.quadrature_weights[selected],
        adjacency_edges=np.asarray(edges, dtype=np.int64),
        adjacency_weights=np.asarray(edge_weights, dtype=float),
    )


def build_observation_schedule():
    """Build 64 detector stations with four downward shield views each."""
    axis = np.linspace(0.6, ROOM_SIZE_M - 0.6, 8)
    x_grid, y_grid = np.meshgrid(axis, axis, indexing="xy")
    stations = np.column_stack(
        [
            x_grid.ravel(),
            y_grid.ravel(),
            np.full(x_grid.size, DETECTOR_HEIGHT_M),
        ]
    )
    view_count = DOWNWARD_VIEW_INDICES.size
    return SimpleNamespace(
        detector_positions_xyz=np.repeat(stations, view_count, axis=0),
        fe_indices=np.tile(DOWNWARD_VIEW_INDICES, stations.shape[0]),
        pb_indices=np.tile(DOWNWARD_VIEW_INDICES, stations.shape[0]),
        live_times_s=np.full(stations.shape[0] * view_count, DWELL_TIME_S),
        station_positions_xyz=stations,
    )


def build_shared_kernel(*, shielded):
    """Build a CPU ContinuousKernel through the shared runtime factory."""
    runtime_config = {"source_rate_model": "detector_cps_1m"}
    if shielded:
        runtime_config["shield_transmission_target"] = 0.1
    else:
        runtime_config.update(
            fe_shield_thickness_cm=0.0,
            pb_shield_thickness_cm=0.0,
        )
    model = build_runtime_observation_model(
        runtime_config,
        isotopes=ISOTOPES,
    )
    return continuous_kernel_from_observation_model(
        model,
        obstacle_grid=None,
        use_gpu=False,
    )

In [ ]:
patches = build_floor_grid()
observations = build_observation_schedule()

shielded_response = build_count_response(
    observations,
    patches,
    ISOTOPES,
    build_shared_kernel(shielded=True),
)[:, :, 0]
unshielded_response = build_count_response(
    observations,
    patches,
    ISOTOPES,
    build_shared_kernel(shielded=False),
)[:, :, 0]

print(f"floor patches: {patches.areas_m2.size}")
print(f"observations: {observations.live_times_s.size}")
print(f"response shape: {shielded_response.shape}")

## Synthetic Poisson observations and current MLE

Each historical point source is mapped to the equally nearest active floor patches. Observations are sampled from the shield-aware shared response with a fixed random seed. Both fits use those same count observations; only the ablation response removes Fe/Pb thickness.

In [ ]:
def map_sources_to_floor(patch_grid, sources):
    """Distribute each point-source strength over equally nearest patches."""
    strengths = np.zeros(patch_grid.areas_m2.size, dtype=float)
    floor_xy = patch_grid.centroids_xyz[:, :2]
    for source_x, source_y, strength in sources:
        squared_distance = np.sum(
            (floor_xy - np.asarray([source_x, source_y])) ** 2,
            axis=1,
        )
        nearest = np.flatnonzero(
            np.isclose(
                squared_distance,
                np.min(squared_distance),
                rtol=0.0,
                atol=1.0e-12,
            )
        )
        strengths[nearest] += float(strength) / nearest.size
    return strengths


def fit_scenario(name, sources, *, seed):
    """Sample one scenario and fit shield-aware and no-shield models."""
    truth = map_sources_to_floor(patches, sources)
    background = BACKGROUND_CPS * observations.live_times_s
    expected = background + shielded_response @ truth
    observed = np.random.default_rng(seed).poisson(expected).astype(float)
    config = SurfaceMapConfig(
        max_iterations=20_000,
        tolerance=1.0e-4,
        objective_tolerance=1.0e-6,
        check_interval=20,
    )
    fits = {}
    for label, response in {
        "shield_aware": shielded_response,
        "no_shield_ablation": unshielded_response,
    }.items():
        fits[label] = fit_surface_map_poisson(
            observed,
            response,
            patches.areas_m2,
            patches.adjacency_edges,
            patches.adjacency_weights,
            background=background,
            config=config,
        )

    metrics = {}
    for label, result in fits.items():
        estimate = result.integrated_strengths_cps_1m[:, 0]
        hotspots = cluster_surface_hotspots(
            patches,
            result.densities_cps_1m_m2,
            ISOTOPES,
            threshold_fraction=0.10,
            min_strength_cps_1m=2.0,
        )
        metrics[label] = {
            "relative_l1": float(
                np.sum(np.abs(estimate - truth)) / np.sum(truth)
            ),
            "poisson_deviance": float(result.deviance),
            "converged": bool(result.converged),
            "iterations": int(result.iterations),
            "hotspot_count": len(hotspots),
        }

    return {
        "name": name,
        "sources": sources,
        "truth_strengths": truth,
        "observed_counts": observed,
        "fits": fits,
        "metrics": metrics,
    }

In [ ]:
results = [
    fit_scenario(name, sources, seed=RANDOM_SEED + index)
    for index, (name, sources) in enumerate(SCENARIOS.items())
]

for scenario in results:
    print(f"\n{scenario['name']}")
    for label, metrics in scenario["metrics"].items():
        print(
            f"  {label:20s} "
            f"relative L1={metrics['relative_l1']:.3f}, "
            f"deviance={metrics['poisson_deviance']:.1f}, "
            f"hotspots={metrics['hotspot_count']}, "
            f"converged={metrics['converged']} "
            f"({metrics['iterations']} iterations)"
        )

## Reconstructed surface maps

The cyan crosses show the continuous historical source coordinates. The maps show integrated patch strength, so a source on a patch boundary can be divided across equally nearest cells.

In [ ]:
grid_cells = int(round(ROOM_SIZE_M / GRID_SPACING_M))
figure, axes = plt.subplots(
    len(results),
    3,
    figsize=(12, 11),
    constrained_layout=True,
)
column_titles = ("Truth", "Shield-aware MLE", "No-shield ablation")

for row, scenario in enumerate(results):
    maps = (
        scenario["truth_strengths"],
        scenario["fits"]["shield_aware"]
        .integrated_strengths_cps_1m[:, 0],
        scenario["fits"]["no_shield_ablation"]
        .integrated_strengths_cps_1m[:, 0],
    )
    maximum = max(float(np.max(values)) for values in maps)
    for axis, values, title in zip(
        axes[row],
        maps,
        column_titles,
        strict=True,
    ):
        image = axis.imshow(
            values.reshape(grid_cells, grid_cells),
            origin="lower",
            extent=(0.0, ROOM_SIZE_M, 0.0, ROOM_SIZE_M),
            vmin=0.0,
            vmax=maximum,
            cmap="inferno",
        )
        for source_x, source_y, _strength in scenario["sources"]:
            axis.scatter(
                source_x,
                source_y,
                color="cyan",
                marker="x",
                s=45,
            )
        axis.set_title(title)
        axis.set_xlabel("x [m]")
        axis.set_ylabel("y [m]")
        figure.colorbar(
            image,
            ax=axis,
            label="integrated strength [cps@1m]",
        )
    axes[row, 0].set_ylabel(f"{scenario['name']}\ny [m]")

plt.show()

## Interpretation and limits

The shield-aware model should achieve lower Poisson deviance because it explains the four orientation-dependent views with the same shared physics used by the current estimator. The no-shield fit is an ablation, not an alternative production model.

This notebook intentionally omits the predecessor's handwritten Adam loop, duplicated plotting cells, CSV snapshots, VAE/GAN experiments, and heuristic NMS implementation. Current graph-based hotspot clustering replaces the old NMS cell.